# Teaching Preparation

In [2]:
from utils import *
import os
import random
import json
from student.agent.agent_raspa import RaspaAgent
random.seed(92)
from student.session_manager import *
import numpy as np

## Prepare knowledge

### Introduction

In [9]:
intro = """
I will teach you about RASPA and how to perform various Monte Carlo (MC) simulations using it. I will cover the following topics:

- What is RASPA and what are its applications
- How to setup a simulation using the provided tools and environment
- Monte Carlo (MC) Simulations in general:
	- What is a MC and how to setup RASPA for these simulations
	- What are the available MC moves
    - Prerequisite simulations
- Specific MC simulation examples:
	- MC in one box: total energy, RDF, angle distributions
	- multiple components	
	- MC on a framework: surface area, pore volume, helium void fraction, Rosenbluth value, adsorption at infinite solution, adsorption isotherm, Henry coefficient, adsorption binary mixture, adsorption selectivities

Try not only to memorize the content, but also to understand the underlying principles and how to apply them in practice. 
It is essential to buid and connect different aspects during the teaching. 
Try now to setup a robust knowledge structure with a lot of connections which you can extend during the teaching.

I will provide examples and explanations for each topic.
Dont ask for clarifications unless I ask you explitly (which I will do!).
"""

In [10]:
intermediate = intro + "\n\n IMPORTANT: Some of aspects have been taught already. Integrate the new knowledge below!\n"

# \nIMPORTANT: Integrate this new information into your memory and highlight as much as possible! Test yourself by asking if it is integrated correctly or needs to be highlighted further!\n"

### General

In [8]:
raspa_general = """
RASPA is a classical molecular simulation software specialized on simulations of porous systems and their interactions with liquids or gases.
RASPA allows for different kinds of simulations for various different purposes.
"""
tools_setup = """
To calculate properties with RASPA, FIRST ask you memory.
Important information are always: simulation input details, simulation prerequisites, output analysis.
If prerequisites are needed (e.g. helium void fraction or ideal rosenbluth weight), ALWAYS first concentrate on simulating these. Then, use the results in a next simulation!

For each individual simulation, you need to generate several files with your tools before running the simulation:
1. Molecules:
    - Identify the molecules (gas/liquid) to simulate. 
    - <molecule loader tool>: Automatically generate molecular definition files and corresponding force field and pseudoatoms files for one or multiple molecules.
2. Box / framework:
    - Identify the system for the simulation: empty box or porous material (MOF, zeolite, ...). 
    - If box: specify in the simulation.input file later
    - If material: <framwork loader tool>: Load the structure .cif files. If you cannot load a structure file, ask for it.
3. simulation input file:
    - Identify the goal of the simulation and ask your memory to find the required settings.
    - <input file tool>: based on the template in the tool description, generate a input file using knowledge from your memory!
4. Run the simulation:
    <execute raspa tool>: run the simulation and automatically generate a new, empty folder for the next simulation.
5. Output:
    - <output tool>: parse relevant information from the *.output file and search for the required properties
    - Some simulations generate additional folders with properties. You can inspect them if you want but else you can ignore the content mostly.
"""

### MC theory

In [11]:
raspa_mc_general = """This is a quick overview of Monte Carlo simulations for molecular simulations:
**Monte Carlo simuations**
Monte Carlo (MC) simulations sample the distribution of states of a systems to determine average, thermodynamic properties.
The transitions between the states are random and have to be chosen to be able to reach every possible state of the system.
In practice, these transitions are called MC moves which typically include the translation, rotation, insertion/deletion of particles for NVT, NpT ensembles or GCMC.
The selection of MC moves implicitly defines the ensemble for the system.

**Configurational bias MC (CBMC)**
For ALL molecules with torsions/bends, reinsertion moves are unlikely to succeed. 
CBMC introduces a much more effictive alternative (in RASPA: PartialReinsertionProbability/CBMBProbability) for these insertions.
To use this, the ideal gas Rosenbluth weight is required as an additional input (see prerequisite simulations later) to correctly use these biased moves.
This value is 1 for small molecules and exponentially decreases for molecules with a lot of torsions.
"""

In [12]:
# According to knowledge/manual/parse_manual.ipynb

from knowledge.manual.latex_parsing import parse_tex, TypeAttr
input_files = "knowledge/manual/raw_knowledge/input_files.tex"
input_files_parsed = parse_tex(input_files)

introduction = input_files_parsed[0]
sim_input = input_files_parsed[1]

framework_blacklist = [4,5,7,8,9,10,12,13,14]
moves_blacklist = [2,3,4,7,8,12,27,28,29,30]
properties_blacklist = [8,9,10,11,12,15,17,18,22,23,24,25,26]
box_blacklist = [2]

def filter_children(node, blacklist_index):
    return [n for i,n in enumerate(node.children) if i not in blacklist_index]

def parse_node(node):
    try:
        node_type = f"\n<type>{node.get_attr(TypeAttr).type}</type>"
    except:
        node_type = ""
    if node.has_child():
        content = "\n\n".join([parse_node(child) for child in node.children])
    else:
        content = node.content

    return f"""<name>{node.title}</name>{node_type}\n<description>\n{content}\n</description>
    """

def build_input(i, nodes):
    n = "\n\n".join(["<keyword>\n"+parse_node(x)+"</keyword>" for x in nodes])
    return f"{i}\n{n}"

filtered_duration = [node for node in sim_input.children[1].children]
filtered_properties = [node for node in filter_children(sim_input.children[11], properties_blacklist)]
filtered_moves = [node for node in filter_children(sim_input.children[10], moves_blacklist)]
filtered_box = [node for node in filter_children(sim_input.children[7], box_blacklist)]
filtered_framework = filter_children(sim_input.children[8], framework_blacklist)

duration_input = build_input("Details regarding the duration setting of a simulation input file:", filtered_duration)
moves_input = build_input("This is a list of molecule properties and monte carlo movescan be assigned to each molecule/component in the simulation.input file to specify the simulation:", filtered_moves)
properties_input = build_input("This is a list of properties and their settings that RASPA can calculate in a simualtion. They will produce extra folders with files specifying the format of the property output:", filtered_properties)
box_input = build_input("These are relevant simulation input parameters when using an empty box.", filtered_box)
framework_input = build_input("These are relevant simulation input parameters when using a framework/material via a .cif file.", filtered_framework)

ModuleNotFoundError: No module named 'fibers'

In [ ]:
moves_additional = """\n\n
Here are additional expert annotations for the MC moves to consider:
- TranslationProbability: Translation Probability can be safely used for all simulations in a framework
- RotationProbability: Rotation Probability can be safely used for all simulations in a framework
- CBMCProbability / PartialReinsertionProbability: CBMC Probability can be safely used for all simulations involving molecules with any torsion parameters in the molecule definition file
- ReinsertionProbability: Reinsertion Probability can be safely used for all simulations in a framework
- SwapProbability: Swap Probability can be safely used for all simulations to calculate adsorption isotherms using GCMC simulations
- WidomProbability: WidomProbability can be safely used for all simulations to calculate Henry’s constant, IdealGasRosenbluthWeight, and helium void fraction
- IdentityChangeProbability: IdentitySwapProbability can be safely used for all simulations to calculate adsorption isotherms of mixtures(more than one component) using GCMC simulations
"""

In [ ]:
moves_input += moves_additional

In [ ]:
mc_intermediate = intermediate +"\nNow I will teach you some specifics from the RASPA instruction manual that will be most relevant. Build an understand of these and use connect them later to the examples (especially the monte carlo moves)."

In [ ]:
raspa_input_details = [duration_input, moves_input, properties_input, box_input, framework_input]

In [ ]:
prerequisites_details = """**Prerequite simulations**
Two types of prerequisites are essential to consider before EVERY simulation. 

HeliumVoidFraction: 
ALWAYS if a framework is specified. 
This values need to be added to every framework definition. 
It is simulated by Widom insertions of Helium on the framework.
The HeliumVoidFraction (between 0 and 1) corresponds to the fraction of empty space.
Its value corresponds to the average widom rosenbluth weight in the RASPA output!

IdealGasRosenbluthWeight: 
ALWAYS if CBMC is used (see above).
This values need to be added to every compound definition with CBMC moves.
It is simulated by Widom insertions of the individual compound in a box.
This simulation can have multiple compounds in parallel since Widom insertions only sample the chemical potential without really inserting a particle.
The IdealGasRosenbluthWeight (between 0 and 1) corrects the MC move acceptance probabilities if biasing is used.
Molecules without rotations (e.g. methane) have a value of 1 which is exponentially decreasing to 0 for larger molecules.
Its value corresponds to the average widom rosenbluth weight in the RASPA output!

IMPORTANT: examples for both simulations will be provided later!
"""

### Example Simulation Inputs

In [13]:
def example_simulation(path):
    ex = {
        "goal" : read_file(os.path.join(path, "goal.txt")),
        "input" : read_file(os.path.join(path, "simulation.input")),
        "output" : read_file(os.path.join(path, "output.txt")),
        "pre" : read_file(os.path.join(path, "prerequisite.txt")),
        "annotation" : read_file(os.path.join(path, "annotation.txt")),
    }
    return ex

In [14]:
path = "knowledge/simulations/"

examples = {
    setup : {
        ex_name : example_simulation(os.path.join(path, setup, ex_name)) for ex_name in os.listdir(os.path.join(path, setup))
    }
    for setup in os.listdir(path) if os.path.isdir(os.path.join(path, setup)) and len(os.listdir(os.path.join(path, setup))) > 0 and setup not in ["templates", "general", ".DS_Store"]
}
# {e: list(examples[e].keys()) for e in examples.keys()}

In [15]:
intro_examples = """
Example simulation setups. These will consist of the following parts:
- <goal/> of the simulation and explainations.
- <input/> (A template of the simulation input file)
- <output/> (keywords that are relevant to analyze from the output. These are EXTREMELY IMPORTANT to remember since the RASPA output is challengingly large. IMPORTANT: if this is empty, just ignore it for now)
- <prerequisites/> (These properties need to be provided or calculated in a separate simulation prior to the main simulation! These are EXTREMELY IMPORTANT to remember since these need to be known in advance. Either they need to be externally provided or calculated with a separate, prior simulation. IMPORTANT: if empty, there is nothing required)

IMPORTANT: these are templates. You need to strictly adapt all values except for those in square brackets (for example [molecule_name]).
IMPORTANT: All the details here are very important. Dont miss any information!
IMPORTANT: some properties can be calculated with multiple different approaches. Try to develop an overview of the different simulation goals and techniques in your memory!
IMPORTANT: make sure to highlight if prerequisites could be required, the memory is associated with the calculation of these properties!
IMPORTANT: some examples use CBMC moves, others dont. You should ALWAYS use CBMC moves depending on the compound and not the simulation type! Dont generalize this from the examples but understand when to use it correctly (learned before)
"""


In [16]:
ex_template = """Simulation Template:

<goal>
{goal}
</goal>
<input>
{input}
</input> 
<output>{output}</output>
<prerequisites>{prerequisites}</prerequisites>
"""


In [17]:
box = examples['mc_box1']
sys = examples["mc_system"]
ex_ordered = [
    box['box1_e'],
    box['box1_rdf'],
    box['box1_density'],
    box['box1_mixture'],
    box['box1_angles'],
    sys["system_hvf"],
    sys["system_rosenbluth"],
    sys["system_surface"],
    sys["system_henry"],
    sys['system_ads_diluted'],
    sys['system_ads_iso'],
    sys['system_ads_n2'],
    sys["system_ads_sel"]
]
ex_input = [ex_template.format(
            goal = ex["goal"],
            input = ex["input"],
            output = ex["output"],
            prerequisites = ex["pre"]
        )
    for ex in ex_ordered
]

In [27]:
print(sys["system_rosenbluth"]["input"])

SimulationType        MonteCarlo
NumberOfCycles        20000
PrintEvery            100
PrintPropertiesEvery  100

Forcefield            local

Box 0
BoxLengths 30 30 30
ExternalTemperature [T]

Component 0 MoleculeName              [molecule name]
            MoleculeDefinition        local
            WidomProbability          1.0
            CreateNumberOfMolecules   0


## Prepare Tasks

### Multi-step

In [17]:
ads_dil = "Determine the adsorption enthalpy of {molecule} on {framework} using a simulation at infinite dilution"
ads_1 = "Determine the adsorption enthalpy of {molecule} on {framework}"
ads_2 = "Compare the adsorption enthalpies of {molecule} and {molecule2} on {framework}"
h = "Determine the henry coefficient of {molecule} on {framework}"
h_2 = "Determine the henry coefficient of {molecule} and {molecule2} on {framework}"

In [18]:
tasks_multistep = [ads_dil, ads_1, ads_2, h, h_2]

### Single-step

In [19]:
add_hvf = " given the helium void fraction of {hvf}"
add_rb_1 = " and the ideal gas rosenbluth weight of {rosenbluth} for {molecule}"
add_rb_2 = " and the ideal gas rosenbluth weight of {rosenbluth} for {molecule} and {rosenbluth2} for {molecule2}"

In [20]:
hvf = "Calculate the helium void fraction of {framework}"
surface = "Determine the surface area of {framework}"
rosenbluth_1 = "Calculate the ideal Rosenbluth weights for {molecule}"
rosenbluth_2 = "Calculate the ideal Rosenbluth weights for {molecule} and {molecule2}"

In [21]:
tasks_framework = [hvf, surface]                                    # framework
tasks_n1 = [rosenbluth_1]                                           # molecule
tasks_n2 = [rosenbluth_2]                                           # molecule, molecule2

tasks_n1_s = [i + add_hvf for i in [ads_dil, ads_1, h]]             # molecule, framework, hvf
tasks_n1_l = [i + add_hvf + add_rb_1 for i in [ads_dil, ads_1, h]]  # molecule, framework, hvf

tasks_n2_ss = [i + add_hvf for i in [ads_2, h_2]]                   # molecule, molecule2, framework, hvf
tasks_n2_sl = [i + add_hvf + add_rb_1 for i in [ads_2, h_2]]                   # molecule, molecule2, framework, hvf
tasks_n2_ll = [i + add_hvf + add_rb_2 for i in [ads_2, h_2]]                   # molecule, molecule2, framework, hvf

### Parameters

In [22]:
molecules_s = ["CO2", "N2", "methane", "ethane"]
molecules_l = ["n-pentane", "n-hexane", "n-heptane"]
rosenbluth = ["0.0197439", "0.0029442", "0.0004450"] # from Aastha

framework = "IRMOF-13"
f_hvf = 0.877

# properties = ["rdf", "density", "RMS distribution", "surface area"]
a="""
Compound	Weight	Error
hexane	0.0029442	0.0000080
pentane	0.0197439	0.0000620
butane	0.1296930	0.0000440
propane	1.0008100	0.0000000
octane	0.0000669	0.0000010
ethane	1.0004100	0.0000000
heptane	0.0004450	0.0000030
"""

In [23]:
instructions_multi = ["Explain all the full procedure to solve this task with your tools (WITHOUT DOING IT): ", "Answer this question using simulations (ALWAYS USE 1/10 cycles and up to 10 molecules for speed. IGNORE the low accuracy!): "]
instructions_single = ["Setup the simulation (WITHOUT EXECUTING IT): ", "Answer this question using simulations (ALWAYS USE 1/10 cycles and up to 10 molecules for speed. IGNORE the low accuracy!): "]
# IMPORTANT: YOU MUST ALWAYS USE 1/10 or less of the amount of simulation steps as your memory tells you and only up to 8 molecules to accelerate the simulation times irrespective of the accuracy!

### Setup

In [24]:
def task_prompt(instruction, task, parameters):
    return instruction + task.format(**parameters)

In [25]:
def task_parameters(r, small=True, mixed = True):

    random.seed(r)
    rx = random.sample(range(len(molecules_l)), 2)

    if mixed is True:
        m1 = molecules_l[rx[0]]
        m2 = molecules_s[rx[1]]
        r1 = rosenbluth[rx[0]]

        parameters = {
            "framework" : framework,
            "hvf" : f_hvf,
            "molecule" : m1,
            "molecule2" : m2,
            "rosenbluth" : r1,
        }
    else:
        if small is True:

            m1 = molecules_s[rx[0]]
            m2 = molecules_s[rx[1]]

            parameters = {
                "framework" : framework,
                "hvf" : f_hvf,
                "molecule" : m1,
                "molecule2" : m2,
            }

        else:

            m1 = molecules_l[rx[0]]
            m2 = molecules_l[rx[1]]
            r1 = rosenbluth[rx[0]]
            r2 = rosenbluth[rx[1]]

            parameters = {
                "framework" : framework,
                "hvf" : f_hvf,
                "molecule" : m1,
                "molecule2" : m2,
                "rosenbluth" : r1,
                "rosenbluth2" : r2
            }
    return parameters

# Teaching

In [26]:
run_id = "b1_matching"

#### Theory teaching

In [ ]:
i = 0

In [ ]:
agent = RaspaAgent(provider="anthropic", path=f"output/{run_id}/", csd_path="./")

In [85]:
def teach(agent, fixed, new_knowledge, i=None):
    if type(new_knowledge) == list:
        for knowledge in new_knowledge:
            agent.reset_chat()
            agent.run(fixed+knowledge)
    else:
        teach(agent, fixed, [new_knowledge], i)
    if i is not None:
        agent.save(f"checkpoints/{run_id}/{i}/")

In [86]:
agent.run(intro)

"I'm ready to learn about RASPA and Monte Carlo simulations! I've established a foundational knowledge structure covering:\n\n**Core Framework:**\n- RASPA as molecular simulation software for MC simulations\n- Applications: framework analysis, multi-component systems, adsorption studies\n- Available tools: framework/molecule loaders, input generators, output parsers\n\n**Learning Structure:**\n1. **RASPA Fundamentals** → Applications & setup methodology\n2. **MC Theory** → Practical implementation & moves\n3. **Simulation Types** → Single box → Multi-component → Framework-based\n4. **Advanced Applications** → Adsorption studies → Selectivity analysis\n\nI'm prepared to build connections between theoretical concepts and practical implementation, understanding both the underlying physics and the technical setup requirements. Please proceed with your detailed explanations and examples - I'll focus on understanding principles rather than just memorizing procedures."

In [93]:
teach(agent, intermediate, [raspa_general, tools_setup], i)

In [94]:
agent.load(f"checkpoints/{run_id}/{i}")

teach(agent, intermediate, [raspa_mc_general])

In [95]:
teach(agent, mc_intermediate, raspa_input_details)

i +=1
teach(agent, intermediate, [prerequisites_details], i)

In [98]:
agent.load(f"checkpoints/{run_id}/{i}")
teach(agent, intermediate, [intro_examples])
teach(agent, intermediate+intro_examples, ex_input)

In [99]:
agent.reset_chat()
i += 1
agent.save(f"checkpoints/{run_id}/{i}/")

In [ ]:
agent.memory_agent.memory.render()

In [71]:
i = 2

#### Run benchmarking

In [190]:
i = 2
benchmarking_id = "benchmark_2"

In [208]:
agent = RaspaAgent(provider="anthropic", path=f"output/{run_id}/", csd_path="./")
agent.load(f"checkpoints/{run_id}/2/")
print("Memory size", agent.memory_size())
print("Average number of keys: ", np.mean([len(k.keys) for k in agent.memory_agent.memory.memory.values()]))
print("Distinct keys: ", len({i for k in agent.memory_agent.memory.memory.values() for i in list(k.keys) }))

Memory size 67
Average number of keys:  7.134328358208955
Distinct keys:  314


In [209]:
agent = RaspaAgent(provider="anthropic", path=f"output/{run_id}/", csd_path="./")
agent.load(f"checkpoints/{run_id}/3/")
print("Memory size", agent.memory_size())
print("Average number of keys: ", np.mean([len(k.keys) for k in agent.memory_agent.memory.memory.values()]))
print("Distinct keys: ", len({i for k in agent.memory_agent.memory.memory.values() for i in list(k.keys) }))

Memory size 91
Average number of keys:  7.78021978021978
Distinct keys:  403


##### Preparation

In [105]:
from student.agent.agent_memory import *

class OfflineLearn(Learn):
    def __init__(self, learning_path:str):
        super().__init__(agent=None)
        self.learning_path = learning_path

    def run(self, context:str):
        self._store(context)
        return tool_response(self.name, "Stored for future learning.")

    def _store(self, context:str):
        """
        Store the context in a file for later learning.
        This is used to store information that can be learned later.
        """
        file = self.learning_path
        os.makedirs(os.path.dirname(file), exist_ok=True)
        with open(file, "a") as f:
            f.write(context + "\n")
        #print(f"Stored context in {file} for future learning.")

In [106]:
prompt = """Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt file in the end explaining the steps. Learn new insights from the simulations.
Task: """

In [107]:
from student.session_manager import *

In [108]:
def bench(task, j, max_iter=20):
    session_id = f"{run_id}/{benchmarking_id}/{j}"
    create_session(session_id=session_id, agent_type="RASPA", provider = "anthropic")
    session = load_session(session_id)
    agent = load_agent(session)
    agent.load(f"checkpoints/{run_id}/{i}/")
    agent.tools["learn"] = OfflineLearn(learning_path=f"offline_learning/{session_id}/task_{j}.txt")
    agent.active_learning = True
    agent.reset_chat()
    agent.auto_run = True
    response = agent.run(task, max_iter=max_iter)

    save_agent(session, agent, note=task)
    
    save_session(session_id=session_id, state = session)
    # print(response)
    return response

##### Single

In [27]:
tasks_single = [
    (tasks_framework, "small"),
    (tasks_n1_s, "small"),
    (tasks_n2_ss, "small"),

    (tasks_n2_sl, "mixed"),

    (tasks_n1, "large"),
    (tasks_n2, "large"),
    (tasks_n1_l, "large"),
    (tasks_n2_ll, "large")
]

In [28]:
tasks_single

[(['Calculate the helium void fraction of {framework}',
   'Determine the surface area of {framework}'],
  'small'),
 (['Determine the adsorption enthalpy of {molecule} on {framework} using a simulation at infinite dilution given the helium void fraction of {hvf}',
   'Determine the adsorption enthalpy of {molecule} on {framework} given the helium void fraction of {hvf}',
   'Determine the henry coefficient of {molecule} on {framework} given the helium void fraction of {hvf}'],
  'small'),
 (['Compare the adsorption enthalpies of {molecule} and {molecule2} on {framework} given the helium void fraction of {hvf}',
   'Determine the henry coefficient of {molecule} and {molecule2} on {framework} given the helium void fraction of {hvf}'],
  'small'),
 (['Compare the adsorption enthalpies of {molecule} and {molecule2} on {framework} given the helium void fraction of {hvf} and the ideal gas rosenbluth weight of {rosenbluth} for {molecule}',
   'Determine the henry coefficient of {molecule} an

In [111]:
random_seeds = [x for x in range(100, 200)]

In [83]:
k = 0

output_single = {}
output_file_single = f"output/{run_id}/{benchmarking_id}/output_single.json"

for instruction in instructions_single:
    for tasks, mol_type in tasks_single:
        parameters = task_parameters(random_seeds[k], small=(mol_type == "small"), mixed=(mol_type == "mixed")) 
        for t in tasks:
            k +=1
            
            task = prompt+task_prompt(instruction, t, parameters)
            print(k, task)
            response = bench(task, k)

            output_single[k] = {"task": task, "response": response}
            with open(output_file_single, "w") as f:
                json.dump(output_single, f)

if not os.path.exists(os.path.dirname(output_file_single)):
    os.makedirs(os.path.dirname(output_file_single))
with open(output_file_single, "w") as f:
    json.dump(output_single, f)

1 Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt file in the end explaining the steps.
Task:
Setup the simulation (WITHOUT EXECUTING IT): Calculate the helium void fraction of IRMOF-13
A CSD path is required to access the coremof files.
Using device: cpu
CIF Name: ./sessions/b1_matching/benchmark_2/1/raspa_output/simulation_1/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/1/raspa_output/simulation_1/framework_pacman.cif
RASPA UnitCells: 2 2 1


[03:20:40] UFFTYPER: Unrecognized atom type: He+4 (0)


2 Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt file in the end explaining the steps.
Task:
Setup the simulation (WITHOUT EXECUTING IT): Determine the surface area of IRMOF-13
A CSD path is required to access the coremof files.
CIF Name: ./sessions/b1_matching/benchmark_2/2/raspa_output/simulation_1/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/2/raspa_output/simulation_1/framework_pacman.cif
RASPA UnitCells: 2 2 1


[03:22:21] UFFTYPER: Unrecognized atom type: Ar3+4 (0)


3 Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt file in the end explaining the steps.
Task:
Setup the simulation (WITHOUT EXECUTING IT): Determine the adsorption enthalpy of CO2 on IRMOF-13 using a simulation at infinite dilution given the helium void fraction of 0.877
A CSD path is required to access the coremof files.
CIF Name: ./sessions/b1_matching/benchmark_2/3/raspa_output/simulation_1/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/3/raspa_output/simulation_1/framework_pacman.cif
RASPA UnitCells: 2 2 1
4 Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt file in the end explaining the steps.
Task:
Setup the simulation 

...

CIF Name: ./sessions/b1_matching/benchmark_2/17/raspa_output/simulation_5/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/17/raspa_output/simulation_5/framework_pacman.cif
RASPA UnitCells: 2 2 1


[03:53:34] UFFTYPER: Unrecognized atom type: He+4 (0)


18 Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt file in the end explaining the steps.
Task:
Answer this question using simulations (ALWAYS USE 1/10 cycles and up to 10 molecules for speed. IGNORE the low accuracy!): Determine the surface area of IRMOF-13
A CSD path is required to access the coremof files.
CIF Name: ./sessions/b1_matching/benchmark_2/18/raspa_output/simulation_1/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/18/raspa_output/simulation_1/framework_pacman.cif
RASPA UnitCells: 2 2 1


[03:54:46] UFFTYPER: Unrecognized atom type: He+4 (0)
[03:56:15] UFFTYPER: Unrecognized atom type: Ar3+4 (0)


19 Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt file in the end explaining the steps.
Task:
Answer this question using simulations (ALWAYS USE 1/10 cycles and up to 10 molecules for speed. IGNORE the low accuracy!): Determine the adsorption enthalpy of methane on IRMOF-13 using a simulation at infinite dilution given the helium void fraction of 0.877
A CSD path is required to access the coremof files.
CIF Name: ./sessions/b1_matching/benchmark_2/19/raspa_output/simulation_1/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/19/raspa_output/simulation_1/framework_pacman.cif
RASPA UnitCells: 2 2 1
CIF Name: ./sessions/b1_matching/benchmark_2/19/raspa_output/simulation_2/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: 

In [113]:
## Redo for this with CO2

output_file_single = f"output/{run_id}/{benchmarking_id}/output_single.json"
if os.path.exists(output_file_single):
    with open(output_file_single, "r") as f:
        output_single = json.load(f)

k = 0
for instruction in instructions_single:
    for tasks, mol_type in tasks_single:
        parameters = task_parameters(random_seeds[k], small=(mol_type == "small"), mixed=(mol_type == "mixed")) 
        for t in tasks:
            k +=1

            if k not in [22,23]:
                continue
            
            task = prompt+task_prompt(instruction, t, parameters)
            print(k, task)
            response = bench(task, k)

            output_single[k] = {"task": task, "response": response}
            with open(output_file_single, "w") as f:
                json.dump(output_single, f)


22 Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt file in the end explaining the steps. Learn new insights from the simulations.
Task: Answer this question using simulations (ALWAYS USE 1/10 cycles and up to 10 molecules for speed. IGNORE the low accuracy!): Compare the adsorption enthalpies of CO2 and methane on IRMOF-13 given the helium void fraction of 0.877
A CSD path is required to access the coremof files.
Using device: cpu
CIF Name: ./sessions/b1_matching/benchmark_2/22/raspa_output/simulation_1/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/22/raspa_output/simulation_1/framework_pacman.cif
RASPA UnitCells: 2 2 1
CIF Name: ./sessions/b1_matching/benchmark_2/22/raspa_output/simulation_2/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
N

##### Multi

In [ ]:
output_multi = {}

In [117]:
random_seeds = [x for x in range(200, 3000)]


j = 100
output_file_multi = f"output/{run_id}/{benchmarking_id}/output_multi.json"

if os.path.exists(output_file_multi):
    with open(output_file_multi, "r") as f:
        output_multi = json.load(f)

for instruction in instructions_multi:
    # for small in [True, False]:
    for t in tasks_multistep:
        j +=1
        if j <= 105:
            continue
        random.seed(random_seeds[j])
        small = random.choice([True, False])
    
        task = prompt+task_prompt(instruction, t, task_parameters(random_seeds[j], small=small))
        print(j, task)
        response = bench(task, j)

        output_multi[j] = {"task": task, "response": response}
        with open(output_file_multi, "w") as f:
            json.dump(output_multi, f)

106 Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt file in the end explaining the steps. Learn new insights from the simulations.
Task: Answer this question using simulations (ALWAYS USE 1/10 cycles and up to 10 molecules for speed. IGNORE the low accuracy!): Determine the adsorption enthalpy of n-hexane on IRMOF-13 using a simulation at infinite dilution
A CSD path is required to access the coremof files.
CIF Name: ./sessions/b1_matching/benchmark_2/106/raspa_output/simulation_1/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/106/raspa_output/simulation_1/framework_pacman.cif
RASPA UnitCells: 2 2 1


[09:27:50] UFFTYPER: Unrecognized atom type: He+4 (0)


CIF Name: ./sessions/b1_matching/benchmark_2/106/raspa_output/simulation_2/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/106/raspa_output/simulation_2/framework_pacman.cif
RASPA UnitCells: 2 2 1
CIF Name: ./sessions/b1_matching/benchmark_2/106/raspa_output/simulation_4/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/106/raspa_output/simulation_4/framework_pacman.cif
RASPA UnitCells: 2 2 1
107 Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt file in the end explaining the steps. Learn new insights from the simulations.
Task: Answer this question using simulations (ALWAYS USE 1/10 cycles and up to 10 molecules for speed. IGNORE the low accuracy!): Determin

[09:30:38] UFFTYPER: Unrecognized atom type: He+4 (0)


CIF Name: ./sessions/b1_matching/benchmark_2/107/raspa_output/simulation_3/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/107/raspa_output/simulation_3/framework_pacman.cif
RASPA UnitCells: 2 2 1
108 Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt file in the end explaining the steps. Learn new insights from the simulations.
Task: Answer this question using simulations (ALWAYS USE 1/10 cycles and up to 10 molecules for speed. IGNORE the low accuracy!): Compare the adsorption enthalpies of n-pentane and methane on IRMOF-13
A CSD path is required to access the coremof files.
CIF Name: ./sessions/b1_matching/benchmark_2/108/raspa_output/simulation_1/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and sa

...

CIF Name: ./sessions/b1_matching/benchmark_2/108/raspa_output/simulation_3/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/108/raspa_output/simulation_3/framework_pacman.cif
RASPA UnitCells: 2 2 1
CIF Name: ./sessions/b1_matching/benchmark_2/108/raspa_output/simulation_4/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/108/raspa_output/simulation_4/framework_pacman.cif
RASPA UnitCells: 2 2 1
CIF Name: ./sessions/b1_matching/benchmark_2/108/raspa_output/simulation_5/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/108/raspa_output/simulation_5/framework_pacman.cif
RASPA UnitCells: 2 2 1
109 Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a pre

[09:38:15] UFFTYPER: Unrecognized atom type: He+4 (0)


CIF Name: ./sessions/b1_matching/benchmark_2/109/raspa_output/simulation_2/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/109/raspa_output/simulation_2/framework_pacman.cif
RASPA UnitCells: 2 2 1


[09:39:30] UFFTYPER: Unrecognized atom type: He+4 (0)


CIF Name: ./sessions/b1_matching/benchmark_2/109/raspa_output/simulation_4/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as ./sessions/b1_matching/benchmark_2/109/raspa_output/simulation_4/framework_pacman.cif
RASPA UnitCells: 2 2 1
110 Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt file in the end explaining the steps. Learn new insights from the simulations.
Task: Answer this question using simulations (ALWAYS USE 1/10 cycles and up to 10 molecules for speed. IGNORE the low accuracy!): Determine the henry coefficient of n-heptane and CO2 on IRMOF-13
A CSD path is required to access the coremof files.
CIF Name: ./sessions/b1_matching/benchmark_2/110/raspa_output/simulation_1/framework.cif
Charge Type: DDEC6
Digits: 10
Atom Type: True
Neutral: True
Keep Connect: True
Compelete and save as 

#### Evaluate and give feedback

In [70]:
feedback = """
The following information is curated to update your knowledge in the memory. 
All these points correct your current understanding and knowledge of RASPA simulations.
<feedback>
{feedback}
</feedback>
IMPORTANT: you MUST integrate the feedback into your memory robustly. Verify internally if the knowledge is updated.
"""

In [ ]:
feedback_f1 = [
"For monte carlo moves, RotationProbability has to be included in all MC simulations of molecules that undergo rotational motion. This includes ALL molecules except for single-atom gases like helium or argon.",
"A MC simulation with Widom insertion moves should not include any other MC moves.",
"A MC simulation with Widom insertion moves should not include any initialization cycles.",
"Ideal gas Rosenbluth weights are only required for molecules with torsions. If no torsion is present, the weight is 1 by default and does not require ANY additional simulation.",
"For simulations of an NVT ensembles, you should not mention any external pressure. This is especially the case for calculating the henry coefficient or rosenbluth weight with Widom insertions!",
"Simulations of mixtures should include the composition through the MolFraction keyword. The MolFractions of each compound have to add up to 1. The default value is 0.5 for each component.",
"For MC simulations, the simulation input file should NOT contain the keyword Ensemble.",
"These are parameters that have to be provided for a compound and not for the global system: SwapProbability, RosenbluthWeight",
"For all grand canonical MC (GCMC) simulations, not only include the swap move, but also all other moves that apply: translation, rotation (if molecule has rotational degrees of freedom), reinsertion, identity swap (if a mixture is simulated).",
"For the output analysis: The acceptance rate of a MC move does not provide any insight into the strength of an interaction. It only serves as a measure of sampling efficiency. If the value is almost 0, the MC simulation could not sample the state space sufficiently.",
"To compare the adsorption strength of different compounds, you should compare the adsorption enthalpies (same pressure) or the loadings in a mixture simulation.",
"To calculate the heat/enthalpy of adsorption, use a GCMC simulation (high pressures), NVT (infinite solution, low pressures) or Widom insertions (infinitely low pressures). There is no explicit ComputeHeatOfAdsorption keyword in RASPA but the property is computed automatically.",
"In RASPA, Widom insertion moves use the keyword WidomProbability for each component (not globally!). There is NO keyword ProbabilityWidomMove",
"For GCMC simulations targeting the calculation of adsorption isotherms, select the pressure range" # whoopsie this was here by mistake
]
# print("\n- ".join(feedback_f1))

In [ ]:
print("\n<next_separate_feedback>\n".join(feedback_f1))

In [115]:
def run_feedback(prompt, session_id, session_old, max_iter=5):

    print("Initializing new session:", session_id)
    
    # Initialize  new agent session
    create_session(session_id=session_id, agent_type="RASPA", provider = "anthropic")
    session = load_session(session_id)
    agent = load_agent(session)
    
    # Load memory from last session
    agent.load(session_old)

    print("Loading session: ", session_old)
    print("Memory size before: ", agent.memory_agent.memory_size())

    # Adjust agent configuration
    agent.active_learning = True
    agent.reset_chat()
    agent.auto_run = False

    # Run and save
    response = agent.run(prompt, max_iter=max_iter)
    print(response)
    print("Memory size after: ", agent.memory_agent.memory_size())
    
    save_agent(session, agent, note=prompt)
    save_session(session_id=session_id, state=session)
    print("Session saved:", session_id)
    return response

In [111]:
benchmarking_id = "feedback_f1"

In [122]:
feedback_f1 = [
    "\n<next_separate_feedback>\n".join(feedback_f1[i:i+4]) for i in range(0, len(feedback_f1), 4)
]

In [126]:
feedback_f1

['For monte carlo moves, RotationProbability has to be included in all MC simulations of molecules that undergo rotational motion. This includes ALL molecules except for single-atom gases like helium or argon.\n<next_separate_feedback>\nA MC simulation with Widom insertion moves should not include any other MC moves.\n<next_separate_feedback>\nA MC simulation with Widom insertion moves should not include any initialization cycles.\n<next_separate_feedback>\nIdeal gas Rosenbluth weights are only required for molecules with torsions. If no torsion is present, the weight is 1 by default and does not require ANY additional simulation.',
 'For simulations of an NVT ensembles, you should not mention any external pressure. This is especially the case for calculating the henry coefficient or rosenbluth weight with Widom insertions!\n<next_separate_feedback>\nSimulations of mixtures should include the composition through the MolFraction keyword. The MolFractions of each compound have to add up 

In [127]:
[len(x.split("<next_separate_feedback>")) for x in feedback_f1]

[4, 4, 4, 2]

In [128]:
output_f1 = {}

j = 0
output_file_f1 = f"output/{run_id}/{benchmarking_id}/output_f1.json"

os.makedirs(os.path.dirname(output_file_f1), exist_ok=True)

if os.path.exists(output_file_f1):
    with open(output_file_f1, "r") as file:
        output_f1 = json.load(file)

session_old = f"checkpoints/{run_id}/{i}/"

for f in feedback_f1:
    feedback_prompt = intermediate+feedback.format(feedback=f)
    
    new_j = f"feedback_{j}"
    session_id = f"{run_id}/{benchmarking_id}/{new_j}"
    
    print(new_j, f)
    response = run_feedback(feedback_prompt, session_id, session_old)
    #print(session_id, session_old)

    output_f1[j] = {"feedback": f, "response": response}
    with open(output_file_f1, "w") as file:
        json.dump(output_f1, file)

    session_old = f"sessions/{run_id}/{benchmarking_id}/{new_j}/checkpoints/"

    j += 1

feedback_0 For monte carlo moves, RotationProbability has to be included in all MC simulations of molecules that undergo rotational motion. This includes ALL molecules except for single-atom gases like helium or argon.
<next_separate_feedback>
A MC simulation with Widom insertion moves should not include any other MC moves.
<next_separate_feedback>
A MC simulation with Widom insertion moves should not include any initialization cycles.
<next_separate_feedback>
Ideal gas Rosenbluth weights are only required for molecules with torsions. If no torsion is present, the weight is 1 by default and does not require ANY additional simulation.
Initializing new session: b1_matching/feedback_f1/feedback_0
A CSD path is required to access the coremof files.
Loading session:  checkpoints/b1_matching/2/
Memory size before:  67

Memory size after:  77
Session saved: b1_matching/feedback_f1/feedback_0
feedback_1 For simulations of an NVT ensembles, you should not mention any external pressure. This is 

In [145]:
# save the updated agent as next i

session = load_session(session_id)
agent = load_agent(session)

# Load memory from last session
agent.load("sessions/"+session_id+"/checkpoints")
agent.reset_chat()
agent.memory_agent.memory_size()

A CSD path is required to access the coremof files.


91

In [149]:
i = 3
agent.save(f"checkpoints/{run_id}/{i}")
print(i)

3


#### Run benchmarking again

In [ ]:
benchmarking_id = "benchmark_2b"
i = 3

In [ ]:
prompt = """Solve problems by asking your memory for details. Find correct solutions for all subtasks! If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt file in the end explaining the steps.
Task: """

In [ ]:
def bench(task, j, max_iter=20):
    session_id = f"{run_id}/{benchmarking_id}/{j}"
    create_session(session_id=session_id, agent_type="RASPA", provider = "anthropic")
    session = load_session(session_id)
    agent = load_agent(session)
    agent.load(f"checkpoints/{run_id}/{i}/")
    del agent.tools["learn"]
    agent.active_learning = False
    agent.reset_chat()
    agent.auto_run = True
    response = agent.run(task, max_iter=max_iter)

    save_agent(session, agent, note=task)
    
    save_session(session_id=session_id, state = session)
    # print(response)
    return response

In [42]:
instruction_f1 =  "Answer this question using simulations: "

##### Make jobs: slightly modified tasks!

In [ ]:
ads_dil = "Determine the adsorption enthalpy of {molecule} on {framework} using a simulation at infinite dilution"
ads_1 = "Determine the adsorption enthalpy of {molecule} on {framework} at a pressure of 1e5 and 300 Kelvin"
ads_2 = "Compare the adsorption enthalpies of a 1:1 mixture of {molecule} and {molecule2} on {framework} at a pressure of 1e5 and 300 Kelvin"
h = "Determine the henry coefficient of {molecule} on {framework}"
h_2 = "Determine the henry coefficient of {molecule} and {molecule2} on {framework}"
# the rest is copied and excluded here. See run_benchmarking2b.py

In [136]:
def ensure_dir(path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)

def load_json(path: str):
    if os.path.exists(path):
        with open(path, "r") as fh:
            try:
                return json.load(fh)
            except json.JSONDecodeError:
                return {}
    return {}

def write_json(path: str, data: dict):
    ensure_dir(path)
    with open(path, "w") as fh:
        json.dump(data, fh, ensure_ascii=False, indent=2)

def build_single_jobs(prompt, instruction):
    # same structure as your original
    tasks_single = [
        (tasks_framework, "small"),
        (tasks_n1_s, "small"),
        (tasks_n2_ss, "small"),
        (tasks_n2_sl, "mixed"),
        (tasks_n1, "large"),
        (tasks_n2, "large"),
        (tasks_n1_l, "large"),
        (tasks_n2_ll, "large"),
    ]
    random_seeds = [x for x in range(200, 300)]

    k = len(tasks_single)  # keep your offset behavior
    jobs = []
    for task_group, mol_type in tasks_single:
        for t in task_group:
            # seed index is current k; id is k+1 (matches your original pattern)
            parameters = task_parameters(
                random_seeds[k],
                small=(mol_type == "small"),
                mixed=(mol_type == "mixed"),
            )
            k += 1
            task_text = prompt + task_prompt(instruction, t, parameters)
            jobs.append({
                "id": k,
                "kind": "single",
                "task": task_text,
            })
    return jobs

def build_multi_jobs(prompt, instruction):
    random_seeds = [x for x in range(300, 3000)]
    base_j = 100

    jobs = []
    for offset, t in enumerate(tasks_multistep):
        j = base_j + offset
        random.seed(random_seeds[j])
        small = random.choice([True, False])

        task_text = prompt + task_prompt(
            instruction,
            t,
            task_parameters(random_seeds[j], small=small)
        )
        jobs.append({
            "id": j,
            "kind": "multi",
            "task": task_text,
        })
    return jobs

def run_one(job):
    # keep stdout behavior similar
    print(job["id"], job["task"])
    resp = bench(job["task"], job["id"])
    return job["id"], job["kind"], job["task"], resp


In [137]:
prompt = (
    "Solve problems by asking your memory for details. Find correct solutions for all subtasks! "
    "If a prerequisite is missing, the result is incorrect. IMPORTANT: ALWAYS add a readme.txt "
    "file in the end explaining the steps. Learn new insights from the simulations.\nTask: "
)
instruction_f1 = "Answer this question using simulations: "

# Prepare jobs
single_jobs = build_single_jobs(prompt, instruction_f1)
multi_jobs  = build_multi_jobs(prompt, instruction_f1)
all_jobs = single_jobs + multi_jobs

# Output files
output_file_single = f"output/{run_id}/{benchmarking_id}/output_single.json"
output_file_multi  = f"output/{run_id}/{benchmarking_id}/output_multi.json"

# Load prior results (merge-on-complete)
output_single = load_json(output_file_single)
output_multi  = load_json(output_file_multi)

# Ensure output dirs exist
ensure_dir(output_file_single)
ensure_dir(output_file_multi)


In [140]:
# run_benchmarking2b.py

In [163]:
# cost
import pandas as pd
df = pd.read_json("output/b1_matching/benchmark_2b/costs.jsonl", lines=True)
df.dropna(subset=['cost'], inplace=True)
# Extract token counts from the nested structure
df['input_tokens'] = df['cost'].apply(lambda x: x['input_tokens'])
df['output_tokens'] = df['cost'].apply(lambda x: x['output_tokens'])

# Group by agent_id and sum tokens
token_sums = df.groupby('run_id').agg({
    'input_tokens': 'sum',
    'output_tokens': 'sum'
}).reset_index()
token_sums

def cost_estimation(tokens, prizes_per_mil):
    total_cost = tokens * prizes_per_mil / 1000000
    return total_cost

def cost_estimate_input(input_tokens):
    return cost_estimation(int(input_tokens), 3)

def cost_estimate_output(output_tokens):
    return cost_estimation(int(output_tokens), 12)

# Calculate costs for each group
token_sums['input_cost'] = token_sums['input_tokens'].apply(cost_estimate_input)
token_sums['output_cost'] = token_sums['output_tokens'].apply(cost_estimate_output)
token_sums['total_cost'] = token_sums['input_cost'] + token_sums['output_cost']
token_sums

,run_id,input_tokens,output_tokens,input_cost,output_cost,total_cost
0,b1_matching,2103836,110735,6.311508,1.32882,7.640328
